# Petals to the Metal -- Flower Classification on TPU

## **โจทย์:**
- จำแนกรูปดอกไม้ `104` ชนิด 
- วัดผลด้วย `Macro F1-score`
- คอลัมน์ `id`, `label`

## **Step 1**: imports, config, find TFRecord files

### Imports

In [1]:
import os, re
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


### Using CPU temporarily

In [2]:
strategy = tf.distribute.get_strategy()
REPLICAS = strategy.num_replicas_in_sync # คำนวณ batch size
AUTO = tf.data.AUTOTUNE # ให้ tf เลือกจำนวน thread ที่ใช้อ่านข้อมูลเอง

print("REPLICAS:", REPLICAS)

REPLICAS: 1


### Config

In [3]:
IMAGE_SIZE = [224, 224]   # ใช้ 224 ตอนทดลองบน CPU
EPOCHS = 12
BATCH_SIZE = 16 * REPLICAS
NUM_CLASSES = 104

KAGGLE_PATH = "/kaggle/input/tpu-getting-started"
LOCAL_PATH = "data/tpu-getting-started"
DATA_PATH = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH
assert os.path.exists(DATA_PATH), "ไม่เจอข้อมูล — กด Add Input แล้วเลือก Petals to the Metal"

PATH = f"{DATA_PATH}/tfrecords-jpeg-{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}"
print(PATH)

data/tpu-getting-started/tfrecords-jpeg-224x224


In [4]:
TRAINING_FILENAMES   = tf.io.gfile.glob(PATH + "/train/*.tfrec")
VALIDATION_FILENAMES = tf.io.gfile.glob(PATH + "/val/*.tfrec")
TEST_FILENAMES       = tf.io.gfile.glob(PATH + "/test/*.tfrec")

assert TRAINING_FILENAMES, "ไม่เจอไฟล์ train — เช็ก PATH"
print("ตัวอย่างชื่อไฟล์:", os.path.basename(TRAINING_FILENAMES[0]))

def count_data_items(filenames):
    # ชื่อไฟล์เป็นแบบ 00-224x224-798.tfrec → ตัวเลขก่อน ".tfrec" คือจำนวนรูปในไฟล์
    return sum(int(re.search(r"-([0-9]+)\.", f).group(1)) for f in filenames)

NUM_TRAINING_IMAGES   = count_data_items(TRAINING_FILENAMES)
NUM_VALIDATION_IMAGES = count_data_items(VALIDATION_FILENAMES)
NUM_TEST_IMAGES       = count_data_items(TEST_FILENAMES)
STEPS_PER_EPOCH = NUM_TRAINING_IMAGES // BATCH_SIZE

print(f"train: {len(TRAINING_FILENAMES)} ไฟล์, {NUM_TRAINING_IMAGES} รูป")
print(f"val  : {len(VALIDATION_FILENAMES)} ไฟล์, {NUM_VALIDATION_IMAGES} รูป")
print(f"test : {len(TEST_FILENAMES)} ไฟล์, {NUM_TEST_IMAGES} รูป")
print("steps per epoch:", STEPS_PER_EPOCH)

ตัวอย่างชื่อไฟล์: 09-224x224-798.tfrec
train: 16 ไฟล์, 12753 รูป
val  : 16 ไฟล์, 3712 รูป
test : 16 ไฟล์, 7382 รูป
steps per epoch: 797
